# Notebook 04: Split data

The labeled compounds are divided once into a training part and a holdout, so every later notebook uses the same split.

- Only the labeled compounds are split. No spectra and no model are used here.
- Compounds with similar structures form a **group**, and a whole group goes to one side. Close relatives on both sides would make test scores look better than they are.
- 20% of each family is the **holdout**, touched once at the end. The rest is divided into 5 folds, 3 times, for the cross-validation in notebook 05.
- Output: `data/labels/split.csv`.

## 1. Load the compounds
### Libraries

In [1]:
import random
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina

RDLogger.DisableLog("rdApp.*")

### Labeled compounds
Only compounds with a known family are split.

In [2]:
labels = pd.read_csv("../data/labels/labels.csv")
known = labels[labels.status == "known"][["inchikey", "name", "family", "class"]].reset_index(drop=True)
families = sorted(known.family.unique())
known.shape

(637, 4)

### Add the structures
The structure of each compound (SMILES) comes from `compounds.csv`.

In [3]:
compounds = pd.read_csv("../data/processed/compounds.csv", skipinitialspace=True)
compounds.columns = compounds.columns.str.strip()
compounds["inchikey"] = compounds.inchikey.str.strip()
compounds["smiles"] = compounds.smiles.str.strip()

known = known.merge(compounds[["inchikey", "smiles"]], on="inchikey", how="left")
assert known.smiles.notna().all()
known.head()

,inchikey,name,family,class,smiles
0,LQZHCIPFXVLYCQ-UHFFFAOYSA-N,4-Phenylmethamphetamine,amphetamine-type,stimulant,CNC(C)Cc1ccc(-c2ccccc2)cc1
1,UFRAAIGOXGTHQX-UHFFFAOYSA-N,1-(2-Chloro-3-fluorophenyl)piperazine,stimulant-other,stimulant,Fc1cccc(N2CCNCC2)c1Cl
2,VNZLQLYBRIOLFZ-UHFFFAOYSA-N,1-(2-Methoxyphenyl)piperazine,stimulant-other,stimulant,COc1ccccc1N1CCNCC1
3,VHFVKMTVMIZMIK-UHFFFAOYSA-N,1-(3-Chlorophenyl)piperazine,stimulant-other,stimulant,Clc1cccc(N2CCNCC2)c1
4,KKIMDKMETPPURN-UHFFFAOYSA-N,1-(3-Trifluoromethylphenyl)piperazine,stimulant-other,stimulant,FC(F)(F)c1cccc(N2CCNCC2)c1


## 2. Similarity between compounds
### Molecules

In [4]:
mols = [Chem.MolFromSmiles(s) for s in known.smiles]
assert all(m is not None for m in mols)

### Fingerprints
Each structure becomes a pattern of 2,048 bits that record which small molecular fragments are present (a Morgan fingerprint).

In [5]:
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = [generator.GetFingerprint(m) for m in mols]

### Similarity table
The similarity of two compounds is the share of fragments they have in common, from 0 to 1 (the Tanimoto score).

In [6]:
similarity = np.array([DataStructs.BulkTanimotoSimilarity(fp, fingerprints) for fp in fingerprints])
similarity.shape

(637, 637)

## 3. Groups
### Settings
A group is one centre compound plus every compound whose similarity to the centre is at or above a cutoff (Butina clustering). One cutoff does not suit every family: at 0.5, nitazene forms a single group and could not be split. Each family therefore uses the loosest cutoff that gives at least `MIN_GROUPS` groups, with no group larger than `MAX_SHARE` of the family.

In [7]:
CUTOFFS = [0.5, 0.6, 0.7, 0.8]
MIN_GROUPS = 5
MAX_SHARE = 0.4

### Distances
The clustering function needs distances, not similarities. The distance is `1 - similarity`, listed row by row for the lower half of the table.

In [8]:
n = len(known)
distances = [1 - similarity[i, j] for i in range(1, n) for j in range(i)]

### Groups at each cutoff
`make_groups` returns a group number for each compound. It is run once per cutoff.

In [9]:
def make_groups(cutoff):
    clusters = Butina.ClusterData(distances, n, 1 - cutoff, isDistData=True)
    group = np.zeros(n, dtype=int)
    for number, members in enumerate(clusters):
        group[list(members)] = number
    return group


groups_at = {cutoff: make_groups(cutoff) for cutoff in CUTOFFS}

### Choose the cutoff of a family
The cutoffs are tried from the loosest (0.5) to the tightest (0.8). The first one that meets both rules is used. If none does, the tightest is used.

In [10]:
def choose_cutoff(rows):
    for cutoff in CUTOFFS:
        sizes = pd.Series(groups_at[cutoff][rows]).value_counts()
        enough_groups = len(sizes) >= MIN_GROUPS
        small_enough = sizes.max() <= MAX_SHARE * len(rows)
        if enough_groups and small_enough:
            return cutoff
    return CUTOFFS[-1]

### Name the groups
Each compound gets the cutoff of its family and a group name such as `fentanyl-type_0.7_3`. The cutoff is saved with the compound.

In [11]:
known["cutoff"] = np.nan
known["group"] = ""

for family, rows in known.groupby("family").groups.items():
    cutoff = choose_cutoff(rows)
    ids = groups_at[cutoff][rows]
    known.loc[rows, "cutoff"] = cutoff
    known.loc[rows, "group"] = [f"{family}_{cutoff}_{i}" for i in ids]

### Groups per family

In [12]:
summary = known.groupby("family").agg(
    compounds=("name", "size"), groups=("group", "nunique"),
    largest_group=("group", lambda g: g.value_counts().max()), cutoff=("cutoff", "first"))
summary

,compounds,groups,largest_group,cutoff
family,,,,
amphetamine-type,33,13,10,0.5
arylcyclohexylamine,30,10,9,0.6
benzodiazepine,37,13,13,0.5
cannabinoid-other,15,8,6,0.5
carboxamide-ester-SC,44,16,13,0.6
cathinone,38,15,10,0.6
depressant-other,24,18,3,0.5
fentanyl-type,51,30,16,0.7
indole-ketone-SC,32,14,8,0.7


## 4. Holdout
### Settings
For each family, the holdout target is 20% of its compounds (at least 2). Whole groups are drawn at random, 300 times, and the draw closest to the target is kept. Ties are broken at random. A family never gives up all its groups. Each family has its own random generator, so a change in one family does not move another.

In [13]:
HOLDOUT_SHARE = 0.20
TRIES = 300
SEED_HOLDOUT = 0

### Draw the holdout groups of a family

In [14]:
def pick_holdout_groups(sizes, rng):
    target = max(2, round(HOLDOUT_SHARE * sizes.sum()))
    names = list(sizes.index)
    assert len(names) >= 2
    best_score, best_pick = None, None
    for _ in range(TRIES):
        how_many = rng.randint(1, min(len(names) - 1, max(1, len(names) // 2)))
        pick = rng.sample(names, how_many)
        total = sizes[pick].sum()
        if total < 2:
            continue
        score = (abs(total - target), rng.random())
        if best_score is None or score < best_score:
            best_score, best_pick = score, pick
    return best_pick

### Apply to every family
Each family draws with its own generator, seeded from the seed and the family name. The draw is reproducible, and a label change in one family leaves the other families as they are.

In [15]:
holdout_groups = []
for family in families:
    sizes = known[known.family == family].groupby("group").size()
    rng = random.Random(f"{SEED_HOLDOUT}-{family}")
    holdout_groups += pick_holdout_groups(sizes, rng)

known["set"] = np.where(known.group.isin(holdout_groups), "holdout", "train")
known.set.value_counts()

set
train      511
holdout    126
Name: count, dtype: int64

## 5. Cross-validation folds
### Settings
The training part is divided into 5 folds. This is done 3 times with different seeds.

In [16]:
N_FOLDS = 5
SEEDS = [0, 1, 2]

### Assign the groups of one repeat to folds
Within each family, the groups are shuffled, then sorted from largest to smallest. Each group goes to the fold that holds the fewest compounds of that family so far (ties at random). This keeps the folds balanced, and a group is never split. Each family has its own generator, seeded from the seed and the family name.

In [17]:
def assign_folds(train, seed):
    fold_of_group = {}
    for family in families:
        rng = random.Random(f"{seed}-{family}")
        sizes = train[train.family == family].groupby("group").size()
        groups = list(sizes.items())
        rng.shuffle(groups)
        groups.sort(key=lambda g: -g[1])
        load = [0] * N_FOLDS
        for group, size in groups:
            fold = min(range(N_FOLDS), key=lambda f: (load[f], rng.random()))
            fold_of_group[group] = fold
            load[fold] += size
    return fold_of_group

### Apply to the three repeats
Holdout compounds get no fold.

In [18]:
train = known[known.set == "train"]

for repeat, seed in enumerate(SEEDS):
    fold_of_group = assign_folds(train, seed)
    known[f"fold_{repeat}"] = known.group.map(fold_of_group).astype("Int64")

known.groupby("fold_0").size()

fold_0
0    117
1     93
2    101
3     99
4    101
dtype: int64

## 6. Checks
### Set up

In [19]:
holdout = known[known.set == "holdout"]
train = known[known.set == "train"]

### Every compound has one role

In [20]:
assert known.inchikey.is_unique
assert known.set.isin(["holdout", "train"]).all()

### No group on both sides of the holdout

In [21]:
assert set(holdout.group).isdisjoint(train.group)

### Every family has at least 2 holdout compounds

In [22]:
assert (holdout.family.value_counts().reindex(families).fillna(0) >= 2).all()

### Each group sits in one fold, and the holdout has none

In [23]:
for repeat in range(len(SEEDS)):
    col = f"fold_{repeat}"
    assert (train.groupby("group")[col].nunique() == 1).all()
    assert holdout[col].isna().all()

### Every family is in the training part of every fold
A family missing from the test part of a fold only prints a warning.

In [24]:
for repeat in range(len(SEEDS)):
    col = f"fold_{repeat}"
    for fold in range(N_FOLDS):
        train_part = train[train[col] != fold]
        test_part = train[train[col] == fold]
        assert set(families) <= set(train_part.family)
        missing = set(families) - set(test_part.family)
        if missing:
            print(f"WARNING: repeat {repeat}, fold {fold}: no test compound from {sorted(missing)}")

## 7. Save and summarize
### Save

In [25]:
split = known[["inchikey", "name", "family", "class", "group", "cutoff", "set", "fold_0", "fold_1", "fold_2"]]
split.to_csv("../data/labels/split.csv", index=False)

### Summary per family

In [26]:
summary["holdout"] = holdout.groupby("family").size()
summary[["compounds", "holdout", "groups", "largest_group", "cutoff"]]

,compounds,holdout,groups,largest_group,cutoff
family,,,,,
amphetamine-type,33,7,13,10,0.5
arylcyclohexylamine,30,6,10,9,0.6
benzodiazepine,37,7,13,13,0.5
cannabinoid-other,15,3,8,6,0.5
carboxamide-ester-SC,44,9,16,13,0.6
cathinone,38,8,15,10,0.6
depressant-other,24,5,18,3,0.5
fentanyl-type,51,10,30,16,0.7
indole-ketone-SC,32,6,14,8,0.7


### Sizes

In [27]:
print(f"holdout: {len(holdout)} of {len(known)} compounds ({len(holdout) / len(known):.1%})")
for repeat in range(len(SEEDS)):
    print(f"repeat {repeat} fold sizes:", train.groupby(f"fold_{repeat}").size().tolist())

holdout: 126 of 637 compounds (19.8%)
repeat 0 fold sizes: [117, 93, 101, 99, 101]
repeat 1 fold sizes: [91, 103, 111, 102, 104]
repeat 2 fold sizes: [86, 102, 109, 107, 107]
